<a href="https://colab.research.google.com/github/gph05010/Medication-Info-Text-Classification/blob/main/02_%EC%9D%B8%EC%82%AC%EC%9D%B4%ED%8A%B8_%EB%8F%84%EC%B6%9C_%EB%B0%8F_%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /content/drive/MyDrive/해커톤

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

In [ ]:
# 데이터 불러오기

DATA_PATH = "./medicine_raw.csv"
df = pd.read_csv(DATA_PATH)

df

In [ ]:
df.info()

In [ ]:
missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isnull().sum().values,
    "missing_ratio": (df.isnull().mean().values * 100).round(2),
    "non_missing_count": df.notnull().sum().values
})

missing_summary = missing_summary.sort_values(
    by="missing_count",
    ascending=False
).reset_index(drop=True)

missing_summary

| 컬럼명                   | 의미            | 예시                                | 프로젝트 활용       |
| --------------------- | ------------- | --------------------------------- | ------------- |
| `entpName`            | 의약품 제조/수입 업체명 | 동화약품(주), 신신제약(주)                  | 참고용           |
| `itemName`            | 의약품명          | 활명수, 아네모정                         | 약품 식별/표시용     |
| `itemSeq`             | 의약품 품목 기준 코드  | 195700020                         | 중복 확인용        |
| `efcyQesitm`          | 효능·효과 설명      | 이 약은 소화불량, 체함에 사용합니다.             | 라벨: 효능        |
| `useMethodQesitm`     | 사용법/복용법 설명    | 성인은 1회 1정, 1일 3회 복용합니다.           | 라벨: 복용법       |
| `atpnWarnQesitm`      | 경고성 주의사항      | 특정 조건에서 복용 금지 등                   | 주의사항 통합 여부 검토 |
| `atpnQesitm`          | 일반 주의사항       | 임부, 어린이, 특정 환자는 의사 또는 약사와 상의하십시오. | 라벨: 주의사항      |
| `intrcQesitm`         | 상호작용 정보       | 특정 약물과 함께 복용하지 마십시오.              | 라벨: 상호작용      |
| `seQesitm`            | 부작용 정보        | 발진, 가려움, 구토 등이 나타날 수 있습니다.        | 라벨: 부작용       |
| `depositMethodQesitm` | 보관 방법         | 습기와 빛을 피해 실온에서 보관하십시오.            | 라벨: 보관법       |
| `openDe`              | 데이터 공개일자      | 20210129                          | 참고용           |
| `updateDe`            | 데이터 수정일자      | 2024-05-09                        | 참고용           |
| `itemImage`           | 의약품 이미지 URL   | `https://nedrug...`               | 이번 모델에서는 미사용  |
| `bizrno`              | 업체 사업자등록번호    | 1108100102                        | 미사용           |


모델 학습용 핵심 텍스트 컬럼
: itemSeq, efcyQesitm, useMethodQesitm, atpnWarnQesitm, atpnQesitm, intrcQesitm, seQesitm, depositMethodQesitm

필요하지만 결측치가 많아 처리해야 하는 컬럼
: atpnWarnQesitm, intrcQesitm, seQesitm

결측치가 조금 있는 컬럼
: atpnQesitm, efcyQesitm, depositMethodQesitm, useMethodQesitm

식별 및 결과 해석용 컬럼
: itemSeq, itemName, entpName

학습에서는 제외하지만 원본 보존할 컬럼
: openDe, updateDe, itemImage, bizrno

In [ ]:
df["atpnWarnQesitm"].dropna().head(10).to_list()

결측치를 임의로 채우지 않는다.

결측치 때문에 행 전체를 삭제하지 않는다.

각 컬럼에서 값이 있는 텍스트만 추출해 학습 데이터로 사용한다.

atpnWarnQesitm은 주의사항에 통합한다.

주의사항 통합하기

In [ ]:
import pandas as pd

# 원본 보존용 복사
df_processed = df.copy()

def merge_warning_and_caution(row):
    warn = row["atpnWarnQesitm"]
    caution = row["atpnQesitm"]

    # NaN 처리 + 문자열 변환
    warn = "" if pd.isna(warn) else str(warn).strip()
    caution = "" if pd.isna(caution) else str(caution).strip()

    # 둘 다 있으면 줄바꿈을 살려서 합치기
    if warn and caution:
        return warn + "\n\n" + caution

    # 경고성 주의사항만 있으면 그것만 사용
    if warn:
        return warn

    # 일반 주의사항만 있으면 그것만 사용
    if caution:
        return caution

    # 둘 다 없으면 결측치로 유지
    return pd.NA

df_processed["atpnQesitm"] = df_processed.apply(
    merge_warning_and_caution,
    axis=1
)

# atpnWarnQesitm 컬럼 삭제
df_processed = df_processed.drop(columns=["atpnWarnQesitm"])

# 확인
df_processed[["itemName", "atpnQesitm"]].head()
df_processed[["itemName", "atpnQesitm"]].tail()

In [ ]:
df_processed[["atpnQesitm"]].isnull().sum()

In [ ]:
df_processed.columns

값이 있는 텍스트만 사용해서 데이터 정리하기

컬럼명을 학습용 라벨링 형태로 바꾸기

In [ ]:
import pandas as pd

label_map = {
    "efcyQesitm": "효능",
    "useMethodQesitm": "복용법",
    "atpnQesitm": "주의사항",
    "intrcQesitm": "상호작용",
    "seQesitm": "부작용",
    "depositMethodQesitm": "보관법"
}

rows = []

for col, label in label_map.items():
    temp = df_processed[["itemSeq", "itemName", col]].copy()

    # 값이 있는 텍스트만 사용
    temp = temp.dropna(subset=[col])

    # 공백만 있는 값 제거
    temp[col] = temp[col].astype(str).str.strip()
    temp = temp[temp[col] != ""]

    # 학습용 컬럼명으로 변경
    temp = temp.rename(columns={col: "text"})
    temp["label"] = label

    rows.append(temp)

df_model = pd.concat(rows, ignore_index=True)

print(df_model.shape)
df_model.tail()

In [ ]:
df_model["label"].value_counts()

In [ ]:
df_model.isnull().sum()

학습을 위한 최소 전처리



앞뒤 공백 제거

여러 줄바꿈 정리

여러 공백 정리

중복 텍스트 제거

너무 짧은 텍스트 제거

In [ ]:
import re
import pandas as pd

def clean_text(text):
    if pd.isna(text):
        return pd.NA

    text = str(text)

    # 앞뒤 공백 제거
    text = text.strip()

    # Windows 줄바꿈 통일
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 문장 끝 마침표 뒤에 바로 한글이 붙은 경우 줄바꿈 추가
    # 예: "보관하십시오.어린이" → "보관하십시오.\n어린이"
    text = re.sub(r"([.!?])(?=[가-힣])", r"\1\n", text)

    # 한국어 종결 표현 뒤에 바로 한글이 붙은 경우도 보정
    # 예: "하십시오.정해진" 같은 케이스는 위에서 처리됨

    # 줄 안의 과도한 공백 정리
    text = re.sub(r"[ \t]+", " ", text)

    # 3개 이상 연속 줄바꿈은 2개로 축소
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

df_model["text_clean"] = df_model["text"].apply(clean_text)

# 비어 있는 값 제거
df_model = df_model.dropna(subset=["text_clean"])
df_model = df_model[df_model["text_clean"].str.len() > 0]

df_model[["text", "text_clean", "label"]].head()

중복 문장 개수 확인

In [ ]:
import pandas as pd
import re

def split_text_to_units(text):
    if pd.isna(text):
        return []

    text = str(text).strip()
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 줄바꿈 기준 분리
    parts = re.split(r"\n+", text)

    # 공백 정리
    parts = [re.sub(r"[ \t]+", " ", p).strip() for p in parts]

    # 너무 짧은 조각 제거
    parts = [p for p in parts if len(p) >= 5]

    return parts

sentence_rows = []

for _, row in df_model.iterrows():
    units = split_text_to_units(row["text_clean"])

    for unit in units:
        sentence_rows.append({
            "itemSeq": row["itemSeq"],
            "itemName": row["itemName"],
            "text": unit,
            "label": row["label"]
        })

df_sentence = pd.DataFrame(sentence_rows)

print(df_sentence.shape)
df_sentence.head()

In [ ]:
dup_check = (
    df_sentence
    .groupby(["text", "label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

dup_check.head(20)

중복 제거하기

In [ ]:
# 문장 단위 데이터셋 기준으로 중복 제거
# 기준: text와 label이 모두 같은 경우만 중복으로 판단

df_sentence_before = df_sentence.copy()

df_sentence_dedup = df_sentence.drop_duplicates(
    subset=["text", "label"]
).reset_index(drop=True)

print("중복 제거 전:", len(df_sentence_before))
print("중복 제거 후:", len(df_sentence_dedup))
print("제거된 행 수:", len(df_sentence_before) - len(df_sentence_dedup))

라벨별로 중복 제거 전후를 비교해보기

In [ ]:
before_counts = df_sentence_before["label"].value_counts()
after_counts = df_sentence_dedup["label"].value_counts()

dedup_compare = pd.DataFrame({
    "before": before_counts,
    "after": after_counts
})

dedup_compare["removed"] = dedup_compare["before"] - dedup_compare["after"]
dedup_compare["removed_ratio"] = (
    dedup_compare["removed"] / dedup_compare["before"] * 100
).round(2)

dedup_compare

중복 제거 결과 보관법 라벨의 데이터가 크게 감소하였다.  
이는 보관법 항목에 “어린이의 손이 닿지 않는 곳에 보관하십시오”, “실온에서 보관하십시오”처럼 정형화된 문장이 반복적으로 등장했기 때문이다.  
따라서 동일한 text-label 조합을 제거하여 특정 반복 문장에 모델이 과하게 의존하는 문제를 줄이고자 했다.

모델 평가에서는 Accuracy만 보지 않고 Macro F1, 라벨별 F1-score를 함께 확인한다.

문장 기준으로 분리한 최종 학습용 데이터

In [ ]:
df_train = df_sentence_dedup[["itemSeq", "itemName", "text", "label"]].copy()

print(df_train.shape)
df_train.head()

In [ ]:
df_train.to_csv(
    "./medicine_sentence_label_dedup.csv",
    index=False,
    encoding="utf-8-sig"
)

df_train.to_excel(
    "./medicine_sentence_label_dedup.xlsx",
    index=False
)

컬럼 기준으로 분리한 최종 학습용 데이터

In [ ]:
# 컬럼 단위 데이터셋 생성
df_column = df_model[["itemSeq", "itemName", "text_clean", "label"]].copy()
df_column = df_column.rename(columns={"text_clean": "text"})

# 비어 있는 값 제거
df_column = df_column.dropna(subset=["text"])
df_column = df_column[df_column["text"].astype(str).str.strip().str.len() > 0]

# text + label 기준 중복 제거
df_column_before = df_column.copy()

df_column = df_column.drop_duplicates(
    subset=["text", "label"]
).reset_index(drop=True)

print("컬럼 단위 중복 제거 전:", len(df_column_before))
print("컬럼 단위 중복 제거 후:", len(df_column))
print("제거된 행 수:", len(df_column_before) - len(df_column))

display(df_column["label"].value_counts())
display(df_column.head())

In [ ]:
df_column.to_csv(
    "./medicine_column_label_dedup.csv",
    index=False,
    encoding="utf-8-sig"
)

df_column.to_excel(
    "./medicine_column_label_dedup.xlsx",
    index=False
)

### 문장 단위 분리 및 중복 제거

원본 의약품 설명문은 줄바꿈을 기준으로 문장 또는 문단이 구분되어 있었다.  
따라서 마침표 기준으로 무리하게 분리하지 않고, 줄바꿈을 기준으로 문장 단위 데이터를 생성하였다.

다만 일부 데이터에서는 `보관하십시오.어린이의 손이...`처럼 문장 사이 줄바꿈이 누락된 경우가 있어, 마침표·물음표·느낌표 뒤에 바로 한글이 이어지는 패턴을 기준으로 줄바꿈을 보정하였다.

문장 단위 분리 후 동일한 안내 문구가 여러 의약품에서 반복되는 것을 확인하였다.  
이에 따라 `text`와 `label`이 모두 동일한 경우를 중복으로 판단하여 제거하였다.  
이 과정은 동일 문장이 반복 학습되어 특정 정형 표현에 모델이 과하게 의존하는 것을 줄이기 위한 처리이다.

### 전처리 결과 요약

의약품 개요정보 데이터는 항목별 설명 컬럼마다 결측치 비율이 달랐다.  
이는 모든 의약품이 효능, 복용법, 주의사항, 상호작용, 부작용, 보관법 정보를 동일하게 갖는 구조가 아니기 때문으로 판단했다.  
따라서 결측치를 임의로 대체하지 않고, 각 설명 컬럼에서 실제 텍스트가 존재하는 경우만 추출하여 text-label 형태의 학습 데이터셋을 구성하였다.

경고성 주의사항인 `atpnWarnQesitm`은 의미상 일반 주의사항인 `atpnQesitm`과 가깝다고 판단하여 주의사항 라벨에 통합하였다.

또한 원본 설명문은 줄바꿈을 기준으로 문장 또는 문단이 구분되어 있어, 줄바꿈 기준 데이터셋과 컬럼 단위 데이터셋을 각각 생성하였다.  
문장 단위 데이터셋에서는 동일한 text-label 조합을 중복으로 판단하여 제거하였다.  
중복 제거 결과 보관법처럼 정형화된 문장이 많은 라벨에서 데이터가 크게 감소했으며, 이는 이후 모델 평가 시 라벨별 F1-score와 Macro F1-score를 함께 확인해야 하는 근거가 된다.